# Блок 11 — multiprocessing: эксперименты

Перед каждым экспериментом — **Прогноз** (до запуска), после — **Вывод** (своими словами).

In [84]:
import os
import random
import time
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import mp_tasks
from mp_tasks import bump, count_words_chunk, count_words_processes, make_chunks

rng = random.Random(42)
vocab = [f"word{i}" for i in range(5_000)]
texts = [" ".join(rng.choices(vocab, k=rng.randint(40, 60))) for _ in range(200_000)]

print("cpu_count:", os.cpu_count())
print("texts:", len(texts), "| words:", sum(len(t.split()) for t in texts))

cpu_count: 8
texts: 200000 | words: 9998149


## Эксперимент 1. Потоки против процессов на CPU-bound

Один и тот же подсчёт слов по 200 000 текстов: последовательно `count_words_chunk(texts)`, `ThreadPoolExecutor(4)` по 4 пачкам из `make_chunks`, `count_words_processes(texts, n_workers=4)`.

### Прогноз

**Какой из трёх вариантов окажется самым быстрым и примерно во сколько раз он быстрее последовательного?** – процессами. Быстрее последовательного примерно в 2 раза.

**Будут ли потоки быстрее последовательного варианта, медленнее или примерно такими же? Почему?** – будут медленнее, потому что под капотом make_chunks и counter: чистый Python код -> потоки будут конкурировать за GIL, и время работы будет дольше последовательного из-за накладных расходов

**Получат ли процессы полное ускорение в 4 раза? Если нет, что его съест?** – сам вопрос уже является подсказкой: полного ускорения в 4 раза не будет, из-за накладных расходов по передаче данных между процессами. Чем больше количество слов в тексте – тем выше разрыв в скорости между последовательным и процессным вариантом

In [ ]:
t0 = time.perf_counter()
res_seq = count_words_chunk(texts)
t_seq = time.perf_counter() - t0

t0 = time.perf_counter()
res_thr = Counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    for part in pool.map(count_words_chunk, make_chunks(texts, 4)):
        res_thr.update(part)
t_thr = time.perf_counter() - t0

t0 = time.perf_counter()
res_proc = count_words_processes(texts, n_workers=4)
t_proc = time.perf_counter() - t0

print(f"последовательно: {t_seq:.2f} с")
print(f"потоки (4):      {t_thr:.2f} с")
print(f"процессы (4):    {t_proc:.2f} с")
print("результаты равны:", res_seq == res_thr == res_proc)

### Вывод

- Первый вопрос – верно
- Второй – Ошибка: правильно – примерно одинаково. GIL я назвал правильно, просто ошибся в оценке накладных расходов: переключений между потоками мало и они дешёвые, поэтому потоки не медленнее последовательного варианта.
- Третий – Верно

## Эксперимент 2. Цена пересылки (`chunksize`)

`ProcessPoolExecutor(4).map(count_words_chunk, ...)` над **одиночными** пачками `[[t] for t in texts[:50_000]]` — при `chunksize=1` и `chunksize=1000`. Плюс последовательный `count_words_chunk` на тех же 50 000 текстах.

### Прогноз

**Расставь три варианта по времени, от самого быстрого к самому медленному.**
1. (Самый быстрый) chunksize=1000
2. Последовательно
3. (Самый медленный) chunksize=1

**Как соотносятся chunksize=1 и последовательный вариант? Дай примерный множитель.** – (последовательный вариант) * 1,5-2 = (chunksize=1)

**Во сколько раз, по-твоему, различаются chunksize=1 и chunksize=1000?** – В 10

In [ ]:
small = texts[:50_000]
singles = [[t] for t in small]

t0 = time.perf_counter()
count_words_chunk(small)
t_seq = time.perf_counter() - t0

timings = {}
for cs in (1, 1000):
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=4) as pool:
        parts = list(pool.map(count_words_chunk, singles, chunksize=cs))
    timings[cs] = time.perf_counter() - t0

print(f"последовательно:          {t_seq:.2f} с")
for cs, t in timings.items():
    print(f"процессы, chunksize={cs:<5}: {t:.2f} с")

### Вывод

- В порядке ошибся, потому что думал, что мы сильно снижаем накладные расходы, поэтому chunksize=1000 и выиграет. Но на практике оказалось, что в принципе – вычисления не сложные и не долгие -> в любом случае последовательный вариант окажется самым быстрым при таком раскладе.
- "chunksize=1 против последовательного" – тут тоже ошибся ровно на 1 порядок –> близко, как и в следующем вопросе

## Эксперимент 3. Изоляция памяти

`bump` увеличивает глобальный `mp_tasks.COUNTER` и возвращает его. Шесть вызовов в пуле из 2 процессов — что вернётся и что окажется в `COUNTER` главного процесса?

### Прогноз

**Что окажется в `results`? Напиши список, который ожидаешь, или опиши, какие значения могут получиться.** – [1, 2, 3, 4, 5, 6]

**Чему будет равен `mp_tasks.COUNTER` в главном процессе после запуска?** – 6

**Будет ли `results` одинаковым от запуска к запуску?** – Нет

In [ ]:
with ProcessPoolExecutor(max_workers=2) as pool:
    results = list(pool.map(bump, range(6)))

print("results:", results)
print("mp_tasks.COUNTER в главном процессе:", mp_tasks.COUNTER)

### Вывод

- **Откуда в запусках 1 и 6 берутся повторы 1 и 2?** – Я не понимаю вообще. Вроде многопроцессность полностью решает проблему гонок данных. Я думал, что на каждом шаге range будет запускаться функция (сразу в двух процессах – значения в процессах одинаковые, так как это просто копия и у каждого отдельный интерпретатор с GIL) и прибавлять одно и то же значение = 1. А тут ещё какие-то повторения
- **Почему в большинстве запусков всё же получается ровно [1..6], хотя процессов два?** – range(6) -> 0, 1, 2, 3, 4, 5 -> к каждому +1 -> 1, 2, 3, 4, 5, 6
- **Почему в главном процессе COUNTER остался 0?** – Не знаю, но интуиция подсказывает, что всё из-за того, что мы смотрим на глобальную переменную, которая равна 0

### Разбор (от Claude, не мой вывод)

- `bump` не использует аргумент: возвращает, сколько раз её вызвали **в этом процессе**.
- При `spawn` каждый воркер заново импортирует `mp_tasks` → у каждого свой `COUNTER = 0`. В `results` — показания двух независимых счётчиков.
- `[1..6]` чаще всего — воркер A стартует первым и забирает все 6 задач, пока B поднимает интерпретатор; совпадение с `range + 1` случайно.
- `[1, 1, 2, 3, 2, 4]`: A взял задачи 0, 2, 3, 5 → 1, 2, 3, 4; B — задачи 1, 4 → 1, 2. `[1, 2, 3, 4, 1, 5]`: B успел взять только задачу 4.
- Это не гонка: повторы — доказательство изоляции памяти. Гонка была бы при записи в одну общую переменную.
- `COUNTER` в главном = 0: воркеры меняли свои копии; назад через pickle приходит только `return`, глобальные переменные не синхронизируются.